In [7]:
from pathlib import Path
import joblib
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
model_path = project_root / "models" / "risk_level_model.joblib"
data_path = project_root / "testing" / "csv_data_saving" / "csv" / "sensor_data.csv"
model_bundle = joblib.load(model_path)
risk_model = model_bundle["model"]
feature_columns = model_bundle["feature_columns"]
max_valid_temperature_c = model_bundle["max_valid_temperature_c"]
temperature_training_max_c = model_bundle["temperature_training_max_c"]

simulation_data = pd.read_csv(data_path)
for column in ["gas_raw", "temperature_c", "humidity_percent"]:
    simulation_data[column] = pd.to_numeric(simulation_data[column], errors="coerce")
simulation_data["recorded_risk"] = (
    simulation_data["threat_level"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .map({"low": "Low", "medium": "Medium", "high": "High"})
    if "threat_level" in simulation_data
    else pd.NA
)
simulation_data = simulation_data.dropna(
    subset=["gas_raw", "temperature_c", "humidity_percent"]
).copy()
if "timestamp" in simulation_data:
    simulation_data["timestamp"] = pd.to_datetime(simulation_data["timestamp"], errors="coerce")
    simulation_data = simulation_data.sort_values(
        "timestamp", kind="stable", na_position="last"
    )
simulation_data = simulation_data.loc[
    simulation_data["gas_raw"].between(0, 4096)
    & simulation_data["temperature_c"].between(-40, max_valid_temperature_c)
    & simulation_data["humidity_percent"].between(0, 100)
].reset_index(drop=True)

if simulation_data.empty:
    raise ValueError("No usable sensor rows found in the source CSV.")

print(f"Loaded model: {model_bundle['model_name']}")
print(f"Replay rows: {len(simulation_data)}")
print(f"Replay temperature range: {simulation_data['temperature_c'].min():.2f} to {simulation_data['temperature_c'].max():.2f} C")
print(f"Training temperature range ended at: {temperature_training_max_c:.2f} C")
print("Ready to run the 5-second replay in the next cell.")

Loaded model: Random Forest
Replay rows: 1202
Replay temperature range: 23.00 to 126.75 C
Training temperature range ended at: 126.75 C
Ready to run the 5-second replay in the next cell.


In [8]:
import math
import time
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix

def run_historical_replay(
    duration_minutes: float = 10.0,
    interval_seconds: float = 5.0,
    start_row: int = 0,
    max_steps: int | None = None,
    save_results: bool = True,
 ):
    """Replay recorded sensor rows at a fixed cadence and compare predictions with labels."""
    if interval_seconds <= 0:
        raise ValueError("interval_seconds must be greater than zero.")
    if duration_minutes <= 0 and max_steps is None:
        raise ValueError("duration_minutes must be positive unless max_steps is provided.")
    if max_steps is not None and max_steps <= 0:
        raise ValueError("max_steps must be positive when provided.")
    if not 0 <= start_row < len(simulation_data):
        raise ValueError("start_row is outside the available replay data.")

    total_steps = (
        math.ceil(duration_minutes * 60 / interval_seconds)
        if max_steps is None
        else max_steps
    )
    results = []
    previous_reading = None
    started_at = datetime.now()
    print(
        f"Starting historical replay: up to {total_steps} readings, "
        f"one every {interval_seconds:g} seconds. Press Ctrl+C to stop."
    )

    try:
        for step in range(total_steps):
            if step > 0:
                time.sleep(interval_seconds)

            row_index = (start_row + step) % len(simulation_data)
            if step > 0 and row_index == 0:
                previous_reading = None
            row = simulation_data.iloc[row_index]
            current = {
                "gas_raw": float(row["gas_raw"]),
                "temperature_c": float(row["temperature_c"]),
                "humidity_percent": float(row["humidity_percent"]),
            }
            features = current.copy()
            if previous_reading is None:
                for sensor in current:
                    features[f"{sensor}_delta"] = 0.0
                    features[f"{sensor}_rate"] = 0.0
            else:
                for sensor, value in current.items():
                    delta = value - previous_reading[sensor]
                    features[f"{sensor}_delta"] = delta
                    features[f"{sensor}_rate"] = delta / interval_seconds

            feature_row = pd.DataFrame([features], columns=feature_columns)
            model_prediction = risk_model.predict(feature_row)[0]
            temperature_out_of_training_range = current["temperature_c"] > temperature_training_max_c
            final_risk = "High" if temperature_out_of_training_range else model_prediction
            probabilities = risk_model.predict_proba(feature_row)[0]
            recorded_risk = row.get("recorded_risk", pd.NA)
            result = {
                "simulated_time": started_at + pd.Timedelta(seconds=step * interval_seconds),
                **current,
                "recorded_risk": recorded_risk,
                "model_prediction": model_prediction,
                "predicted_risk": final_risk,
                "temperature_out_of_training_range": temperature_out_of_training_range,
            }
            result.update({
                f"probability_{label}": probability
                for label, probability in zip(risk_model.classes_, probabilities)
            })
            results.append(result)

            truth_text = recorded_risk if pd.notna(recorded_risk) else "unlabeled"
            override_text = " [temperature fail-safe]" if temperature_out_of_training_range else ""
            print(
                f"{step + 1:04d} | {current['temperature_c']:7.2f} C | "
                f"gas {current['gas_raw']:4.0f} | humidity {current['humidity_percent']:5.1f}% | "
                f"predicted {final_risk:6s} (model {model_prediction:6s}) | "
                f"recorded {truth_text}{override_text}"
            )
            previous_reading = current

    except KeyboardInterrupt:
        print("\nReplay interrupted. Summarizing readings collected so far.")

    results_frame = pd.DataFrame(results)
    if results_frame.empty:
        print("No readings were processed.")
        return results_frame

    print(f"\nReplay finished with {len(results_frame)} readings.")
    print("Predicted risk counts:\n", results_frame["predicted_risk"].value_counts())
    print("Recorded risk counts:\n", results_frame["recorded_risk"].value_counts(dropna=False))
    print(
        "Temperature fail-safe activations:",
        int(results_frame["temperature_out_of_training_range"].sum()),
    )

    labeled_results = results_frame.dropna(subset=["recorded_risk"])
    if not labeled_results.empty:
        print(
            "\nConfusion matrix (rows=recorded, columns=predicted; Low, Medium, High):\n",
            confusion_matrix(
                labeled_results["recorded_risk"],
                labeled_results["predicted_risk"],
                labels=["Low", "Medium", "High"],
            ),
        )
        print(
            "\nReplay classification report:\n",
            classification_report(
                labeled_results["recorded_risk"],
                labeled_results["predicted_risk"],
                labels=["Low", "Medium", "High"],
                zero_division=0,
            ),
        )

    results_frame["risk_changed"] = results_frame["predicted_risk"].ne(
        results_frame["predicted_risk"].shift()
    )
    transitions = results_frame.loc[results_frame["risk_changed"]].copy()
    print("Predicted risk transitions:")
    print(transitions[["simulated_time", "temperature_c", "predicted_risk"]].to_string(index=False))

    if save_results:
        output_path = (
            project_root / "testing" / "csv_data_saving" / "csv"
            / f"risk_replay_{started_at:%Y%m%d_%H%M%S}.csv"
        )
        results_frame.to_csv(output_path, index=False)
        print("Saved replay results:", output_path)

    return results_frame

In [11]:
def simulate_parameter_ramp(
    ramp_parameter: str = "temperature_c",
    starting_value: float = 24.0,
    increase_per_reading: float = 5.0,
    duration_seconds: float = 60.0,
    interval_seconds: float = 5.0,
    gas_raw: float = 300.0,
    temperature_c: float = 24.0,
    humidity_percent: float = 40.0,
 ):
    """Increase temperature or gas_raw once per interval and show model predictions."""
    if ramp_parameter not in {"temperature_c", "gas_raw"}:
        raise ValueError("ramp_parameter must be 'temperature_c' or 'gas_raw'.")
    if increase_per_reading <= 0:
        raise ValueError("increase_per_reading must be greater than zero.")
    if duration_seconds <= 0 or interval_seconds <= 0:
        raise ValueError("duration_seconds and interval_seconds must be greater than zero.")

    sensor_limits = {
        "gas_raw": (0.0, 4096.0),
        "temperature_c": (-40.0, max_valid_temperature_c),
        "humidity_percent": (0.0, 100.0),
    }
    current = {
        "gas_raw": float(gas_raw),
        "temperature_c": float(temperature_c),
        "humidity_percent": float(humidity_percent),
    }
    for sensor, value in current.items():
        lower_limit, upper_limit = sensor_limits[sensor]
        if not lower_limit <= value <= upper_limit:
            raise ValueError(f"{sensor} must be between {lower_limit:g} and {upper_limit:g}.")
    current[ramp_parameter] = float(starting_value)
    lower_limit, upper_limit = sensor_limits[ramp_parameter]
    if not lower_limit <= starting_value <= upper_limit:
        raise ValueError(f"starting_value must be between {lower_limit:g} and {upper_limit:g}.")

    results = []
    previous_reading = None
    reading_number = 0
    started_at = time.monotonic()
    wall_clock_start = datetime.now()
    print(
        f"Ramping {ramp_parameter} from {starting_value:g} by "
        f"{increase_per_reading:g} per reading every {interval_seconds:g}s "
        f"for up to {duration_seconds:g}s. Press Ctrl+C to stop."
    )
    if ramp_parameter == "gas_raw":
        print("gas_raw is being used as a proxy for the smoke sensor signal.")

    try:
        while time.monotonic() - started_at < duration_seconds:
            elapsed = time.monotonic() - started_at
            ramped_value = min(
                starting_value + reading_number * increase_per_reading,
                upper_limit,
            )
            current[ramp_parameter] = ramped_value
            features = current.copy()
            if previous_reading is None:
                for sensor in current:
                    features[f"{sensor}_delta"] = 0.0
                    features[f"{sensor}_rate"] = 0.0
            else:
                for sensor, value in current.items():
                    delta = value - previous_reading[sensor]
                    features[f"{sensor}_delta"] = delta
                    features[f"{sensor}_rate"] = delta / interval_seconds

            feature_row = pd.DataFrame([features], columns=feature_columns)
            model_prediction = risk_model.predict(feature_row)[0]
            probabilities = risk_model.predict_proba(feature_row)[0]
            temperature_out_of_training_range = current["temperature_c"] > temperature_training_max_c
            predicted_risk = "High" if temperature_out_of_training_range else model_prediction
            result = {
                "elapsed_seconds": elapsed,
                "timestamp": wall_clock_start + pd.Timedelta(seconds=elapsed),
                **current,
                "model_prediction": model_prediction,
                "predicted_risk": predicted_risk,
                "temperature_out_of_training_range": temperature_out_of_training_range,
            }
            result.update({
                f"probability_{label}": probability
                for label, probability in zip(risk_model.classes_, probabilities)
            })
            results.append(result)

            print(
                f"t={elapsed:6.1f}s | {ramp_parameter}={ramped_value:7.2f} | "
                f"model={model_prediction:6s} | risk={predicted_risk:6s}"
                + (" [temperature fail-safe]" if temperature_out_of_training_range else "")
            )
            previous_reading = current.copy()
            reading_number += 1
            remaining = duration_seconds - (time.monotonic() - started_at)
            if remaining > 0:
                time.sleep(min(interval_seconds, remaining))

    except KeyboardInterrupt:
        print("\nRamp simulation interrupted.")

    results_frame = pd.DataFrame(results)
    if not results_frame.empty:
        transitions = results_frame.loc[
            results_frame["predicted_risk"].ne(results_frame["predicted_risk"].shift())
        ]
        print("\nRisk counts:\n", results_frame["predicted_risk"].value_counts())
        print("Risk transitions:")
        print(
            transitions[["elapsed_seconds", ramp_parameter, "predicted_risk"]]
            .to_string(index=False)
        )
    return results_frame

## Run a ramp simulation

Choose `temperature_c` or `gas_raw`; each sample increases that one value while the other sensors stay fixed. The function stops after `duration_seconds`, returns a DataFrame, and prints risk changes as they happen. This is a synthetic input test, not a fire-safety or sensor-validation test.

In [13]:
ramp_results = simulate_parameter_ramp(
    ramp_parameter="temperature_c",
    starting_value=24.0,
    increase_per_reading=5.0,
    duration_seconds=60,
    interval_seconds=5,
    gas_raw=300,
    humidity_percent=42.9,
)

Ramping temperature_c from 24 by 5 per reading every 5s for up to 60s. Press Ctrl+C to stop.
t=   0.0s | temperature_c=  24.00 | model=Low    | risk=Low   
t=   5.1s | temperature_c=  29.00 | model=Low    | risk=Low   
t=  10.3s | temperature_c=  34.00 | model=Medium | risk=Medium
t=  15.4s | temperature_c=  39.00 | model=High   | risk=High  
t=  20.5s | temperature_c=  44.00 | model=High   | risk=High  
t=  25.6s | temperature_c=  49.00 | model=High   | risk=High  
t=  30.8s | temperature_c=  54.00 | model=High   | risk=High  
t=  35.9s | temperature_c=  59.00 | model=High   | risk=High  
t=  41.0s | temperature_c=  64.00 | model=High   | risk=High  
t=  46.1s | temperature_c=  69.00 | model=High   | risk=High  
t=  51.3s | temperature_c=  74.00 | model=High   | risk=High  
t=  56.4s | temperature_c=  79.00 | model=High   | risk=High  

Risk counts:
 predicted_risk
High      9
Low       2
Medium    1
Name: count, dtype: int64
Risk transitions:
 elapsed_seconds  temperature_c predicted

In [9]:
# Short smoke test; replace with the long-run settings below when ready.
replay_results = run_historical_replay(
    interval_seconds=0.01,
    max_steps=3,
    save_results=False,
)

Starting historical replay: up to 3 readings, one every 0.01 seconds. Press Ctrl+C to stop.
0001 |   24.00 C | gas 1456 | humidity  42.9% | predicted High   (model High  ) | recorded High
0002 |   24.00 C | gas 1692 | humidity  42.9% | predicted High   (model High  ) | recorded High
0003 |   24.00 C | gas 1360 | humidity  42.9% | predicted High   (model High  ) | recorded High

Replay finished with 3 readings.
Predicted risk counts:
 predicted_risk
High    3
Name: count, dtype: int64
Recorded risk counts:
 recorded_risk
High    3
Name: count, dtype: int64
Temperature fail-safe activations: 0

Confusion matrix (rows=recorded, columns=predicted; Low, Medium, High):
 [[0 0 0]
 [0 0 0]
 [0 0 3]]

Replay classification report:
               precision    recall  f1-score   support

         Low       0.00      0.00      0.00         0
      Medium       0.00      0.00      0.00         0
        High       1.00      1.00      1.00         3

    accuracy                           1.00      

## Run the replay

The replay loops through the recorded CSV in timestamp order, feeding one row to the model every five seconds. Change `duration_minutes` below to extend the run; the readings will loop back to the start of the CSV when needed. Interrupt the cell with the notebook stop button or Ctrl+C to print the summary and save results. The recorded labels are from data already used to fit this model, so the comparison is in-sample and is not an independent measure of accuracy.

In [10]:
replay_results = run_historical_replay(
    duration_minutes=2,
    interval_seconds=5,
    save_results=True,
)

Starting historical replay: up to 24 readings, one every 5 seconds. Press Ctrl+C to stop.
0001 |   24.00 C | gas 1456 | humidity  42.9% | predicted High   (model High  ) | recorded High
0002 |   24.00 C | gas 1692 | humidity  42.9% | predicted High   (model High  ) | recorded High
0003 |   24.00 C | gas 1360 | humidity  42.9% | predicted High   (model High  ) | recorded High
0004 |   23.50 C | gas 1085 | humidity  42.9% | predicted High   (model High  ) | recorded High
0005 |   24.00 C | gas  913 | humidity  42.8% | predicted High   (model High  ) | recorded High
0006 |   23.75 C | gas  810 | humidity  42.8% | predicted High   (model High  ) | recorded High
0007 |   24.25 C | gas  719 | humidity  42.8% | predicted High   (model High  ) | recorded High
0008 |   24.00 C | gas  679 | humidity  42.9% | predicted High   (model High  ) | recorded High
0009 |   24.00 C | gas  603 | humidity  44.2% | predicted High   (model High  ) | recorded High
0010 |   24.00 C | gas  605 | humidity  44.4% 